In [2]:
import numpy as np
import rasterio as rio
from rasterio.warp import reproject, Resampling
from pyproj import CRS
from affine import Affine
from tqdm import tqdm
from matplotlib import pyplot as plt
from pathlib import Path

In [3]:
ROOT = Path.cwd().parent
DATA = ROOT / "data"
RESULTS = ROOT / "results"
FEATURE_DIR = RESULTS / "features"

In [10]:
country_raster_1km_ = rio.open(RESULTS / 'country-rasters' / 'country-raster-eck4-1km.tif')

In [6]:
DATA = Path(r'F:\code\20250117\data')

In [7]:
GLWD_v2_delta_main_class_50pct_ = rio.open(DATA / 'GLWD_v2_delta_combined_classes_tif' / 'GLWD_v2_delta_combined_classes' / 'GLWD_v2_delta_main_class_50pct.tif')

In [8]:
GLWD_v2_delta_main_class_50pct = GLWD_v2_delta_main_class_50pct_.read(1)

In [9]:
k = 4800
I = GLWD_v2_delta_main_class_50pct_.shape[0] // k 
J = GLWD_v2_delta_main_class_50pct_.shape[1] // k 

m = 2
h = GLWD_v2_delta_main_class_50pct_.shape[0] // m 
w = GLWD_v2_delta_main_class_50pct_.shape[1] // m 

n = int(4800 // m)

idx_set = [(i, j) for i in range(I) for j in range(J)] 
mask_values = [i for i in range(34)] + [255] 

GLWD_v2_resampled = np.empty((h, w), dtype='uint8')

for i, j in tqdm(idx_set):
    a_window = ((i*k, (i+1)*k), (j*k, (j+1)*k))
    a_part = GLWD_v2_delta_main_class_50pct_.read(1, window=a_window)
    a_resampled_part = np.empty((n, n), dtype='uint8')
    for gg in mask_values:
        a_mask = (a_part == gg)
        reshaped_mask = a_mask.reshape(k // m, m, k // m, m) 
        count_values = reshaped_mask.sum(axis=(1, 3))
        a_resampled_part[count_values >= 2] = gg
    
    GLWD_v2_resampled[i*n:(i+1)*n, j*n:(j+1)*n] = a_resampled_part


100%|██████████| 126/126 [05:45<00:00,  2.74s/it]


In [11]:
GLWD_v2_resampled_global = np.vstack([np.ones((720, 43200), dtype='uint8') * 255, GLWD_v2_resampled, np.ones((4080, 43200), dtype='uint8') * 255])

In [12]:
print(GLWD_v2_resampled_global.shape)

(21600, 43200)


In [13]:
GLWD_v2_1km = np.empty(country_raster_1km_.shape, dtype=np.uint8)

In [14]:
GLWD_transform = Affine(1/120, 0, -180, 0, -1/120, 90)

In [15]:
reproject(
    source=GLWD_v2_resampled_global, 
    destination=GLWD_v2_1km, 
    src_transform=GLWD_transform, 
    src_crs='EPSG:4326', 
    dst_transform=country_raster_1km_.transform, 
    dst_crs=country_raster_1km_.crs, 
    dst_nodata=255,
    resampling=Resampling.nearest, 
)

(array([[255, 255, 255, ..., 255, 255, 255],
        [255, 255, 255, ..., 255, 255, 255],
        [255, 255, 255, ..., 255, 255, 255],
        ...,
        [255, 255, 255, ..., 255, 255, 255],
        [255, 255, 255, ..., 255, 255, 255],
        [255, 255, 255, ..., 255, 255, 255]],
       shape=(16922, 33844), dtype=uint8),
 Affine(1000.0, 0.0, -16922000.0,
        0.0, -1000.0, 8461000.0))

In [ ]:
water_mask = np.where((GLWD_v2_1km >= 1) & (GLWD_v2_1km <= 7), 1, 0).astype(np.uint8)

In [33]:
with rio.open(
    FEATURE_DIR / 'water-mask-1km.tif',
    "w",
    driver="GTiff",
    height=country_raster_1km_.shape[0],
    width=country_raster_1km_.shape[1],
    count=1,
    dtype=GLWD_v2_1km.dtype,
    crs=country_raster_1km_.crs,
    transform=country_raster_1km_.transform,
    compress='lzw'
) as dst:
    dst.write(water_mask, 1)